# # Install dependencies

In [ ]:
!pip install -q --user transformers datasets accelerate


# # Load training data

In [ ]:
"""Load the training data.

transformers is NLP-oriented — using an inline sentiment dataset for the
demo. `data.split` shuffles rows by default (use `shuffle=False` to keep
order); `random_state` makes the split reproducible.
"""
import pandas as pd
from nubison_model import data

raw = pd.DataFrame({
    "text": [
        "I love this product", "Amazing experience", "Wonderful day",
        "Best ever made", "Excellent quality", "Pretty good overall",
        "Terrible service", "I hate it", "Awful experience",
        "Worst day ever", "Disappointing result", "Not good at all",
    ],
    "target": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

datasets = data.split(raw, {"train": 0.75, "val": 0.25}, random_state=42)
for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


# # Train (HuggingFace transformers)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# A random-weighted tiny DistilBert (~400 KB pickled) keeps the demo's
# mlflow artifact upload small enough to fit any reasonable proxy buffer
# and avoids pulling sentencepiece. Swap to distilbert-base-uncased or
# any other checkpoint in production.
MODEL_ID = "hf-internal-testing/tiny-random-DistilBertForSequenceClassification"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)


def _as_hf_dataset(X, y):
    df = X.assign(labels=y.values.ravel())
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(
        lambda r: tokenizer(r["text"], padding="max_length", truncation=True, max_length=16),
        batched=True,
    )


In [ ]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring).
    target        — label column name(s); single string or list of strings.
    model_type    — "classifier" / "regressor" / free-form string tag.
    weights_path  — default "src/weights.pkl".
"""
from transformers import Trainer, TrainingArguments
from nubison_model import train

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    train_ds = _as_hf_dataset(t.X_train, t.y_train)
    val_ds = _as_hf_dataset(t.X_val, t.y_val)

    args = TrainingArguments(
        output_dir="/tmp/hf_trainer",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        logging_steps=1,
        report_to=["mlflow"],
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()

    t.save(model)

print(f"run_id: {t.run_id}")
